# Task 3 - Fashion Occasion and Gender Classification

**Goal:** predict both catalogue `gender` and `usage` from the same product image.
We retain the existing notebook's two-target MLP approach: five gender labels and eight usage labels, instead of a sparse joint label with up to 40 combinations.

The experiment follows goal -> data audit -> baseline -> improvements -> validation selection -> untouched holdout -> saved-model inference. All models are trained from scratch on the supplied data. Only pixels enter the models; metadata supplies labels and split strata, not unavailable inference features.

**Scope:** this is the Task 3 contribution, not the team's complete report or submission. Gender means the retailer's intended product audience, not a person's identity. Review the code and course AI-disclosure requirements before submitting.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Image as DisplayImage, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / 'A2_FashionDataset/FashionDataset/train/styles_train.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the extracted repository, or set ROOT explicitly.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tasks.task3_gender_usage.data import prepare_image
from tasks.task3_gender_usage.train import Config, run_experiment
from tasks.task3_gender_usage.predict import Task3Predictor, export_submission, notebook_demo

MODE = os.environ.get('TASK3_MODE', 'train')
EXISTING_RUN = os.environ.get('TASK3_RUN_DIR')
CONFIG = Config(epochs=2 if MODE == 'smoke' else 20, smoke=MODE == 'smoke')
print('Python:', sys.executable)
print('Repository:', ROOT)
print('Mode:', MODE)
if MODE not in {'train', 'smoke', 'review'}:
    raise ValueError('MODE must be train, smoke or review.')

## Task Definition and Experiment Design

**Primary score:** fixed-vocabulary macro-F1, with accuracy and per-class support beside it. The model is not successful merely because it predicts the dominant label often.

**Split:** approximately 70% training, 15% validation and 15% internal holdout. Exact decoded-RGB duplicate groups stay in a single partition. Within each joint gender/usage stratum, groups are shuffled with seed 42. Strata with fewer than three groups, and groups with conflicting target labels, remain in training; coverage is reported explicitly. Missing usage does not remove a valid gender example.

Both targets and the merged-label experiment share row assignments. This intentionally replaces the notebook's previous separate random splits. The root README's shared `src` loader no longer exists, so this split is **Task 3-local**, not claimed to match teammates' results.

The supplied test CSV has no true labels and cannot provide measured test accuracy. The internal holdout remains unseen during tuning. It is not a substitute for external real-world validation.

## Data Loading and Initial Audit

Read the source CSV without changing it. Extra `Unnamed` fields are inspected, then excluded from image-only training. Full image checks are performed by the experiment and displayed below, including unreadable files, grayscale, irregular sizes and duplicate-label conflicts.

In [ ]:
DATASET = ROOT / 'A2_FashionDataset/FashionDataset'
raw = pd.read_csv(DATASET / 'train/styles_train.csv')
blind_template = pd.read_csv(DATASET / 'test/styles_prediction.csv')
display(raw.head())
display(raw.isna().sum().rename('missing_values').to_frame())
print('Train metadata rows:', len(raw), '| Blind test rows:', len(blind_template))
junk_columns = [name for name in raw if name.startswith('Unnamed')]
if junk_columns:
    display(raw.loc[raw[junk_columns].notna().any(axis=1), ['id', 'productDisplayName', *junk_columns]].head())

In [ ]:
for target in ('gender', 'usage'):
    counts = raw[target].value_counts(dropna=False)
    display(pd.DataFrame({'count': counts, 'share_percent': (100 * counts / len(raw)).round(2)}))

### Images by Class

These grids are descriptive examples, not external evaluation. A product photograph does not necessarily reveal actual garment size, manufacturer intent or every suitable occasion. Those are reasons to inspect class-specific errors, not proof that a class is impossible to learn.

In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt

def show_samples(frame, target, count=3):
    classes = sorted(frame[target].dropna().unique())
    figure, axes = plt.subplots(len(classes), count, figsize=(6, 1.6 * len(classes)), squeeze=False)
    for row, label in enumerate(classes):
        paths = [DATASET / 'train/images_train' / f'{identity}.jpg' for identity in frame.loc[frame[target].eq(label), 'id']]
        paths = [path for path in paths if path.is_file()][:count]
        for column, axis in enumerate(axes[row]):
            axis.axis('off')
            if column < len(paths):
                axis.imshow(prepare_image(paths[column], (80, 60)))
            if column == 0:
                axis.set_title(label, fontsize=9)
    figure.tight_layout()
    display(figure)
    plt.close(figure)

show_samples(raw, 'gender')
show_samples(raw, 'usage')

## Preprocessing and Model Choice

All images become RGB, resized with bilinear interpolation to **32 high x 24 wide**, preserving the common source aspect ratio. A `Rescaling(1/255)` layer is saved inside the model, so training and prediction cannot accidentally normalize differently. The input is 2,304 pixel values, reducing CPU cost compared with 14,400 at the original resolution. The trade-off is loss of fine detail.

**Important correction:** flattening keeps pixel order and is reversible; it does not erase spatial information. An MLP can learn shapes but has no convolutional locality or shared filters, so it is less sample-efficient and more sensitive to alignment than a CNN. This task retains MLP as requested and does not reuse Task 1 weights.

No labels are guessed or filled from another target. Missing labels are filtered only for that target. Raw pixel scaling is fixed, with no fitted statistics from validation or holdout.

## Baseline and Improvements

| Candidate | Method | Reason and trade-off |
| --- | --- | --- |
| Majority | Always predict the most common **training** label | Exposes misleading accuracy under imbalance; learns no image features |
| Default MLP | Flatten -> Dense(256, sigmoid) -> softmax | Simple starting point; may saturate and underfit |
| Regularized MLP | Flatten -> Dense(256, ReLU) -> Dropout(0.3) -> Dense(128, ReLU) -> Dropout(0.3) -> softmax | More capacity and regularization; more parameters and hyperparameters |
| Weighted MLP | Same regularized architecture + capped square-root inverse-frequency class weights | Gives rare classes a stronger training signal; can reduce majority accuracy and amplify noise |

Each MLP uses Adam (0.001), sparse cross-entropy, batch size 128 and at most 20 epochs. Early stopping monitors **validation macro-F1**, patience 5, and restores the best accepted weights. Class weights use training counts only.

The default-to-regularized comparison changes an architecture bundle, not a single variable. Only the regularized-to-weighted comparison isolates class weighting. Both start from the same random seed. The final original-label MLP is selected by validation macro-F1; exact ties prefer fewer parameters, then a stable model name. Neither architecture nor epoch count is changed after looking at holdout.

## Training

Run All trains and writes a new timestamped directory under `outputs/task3/`; it does not overwrite earlier runs. `MODE='smoke'` checks a subset in two epochs and must not be quoted as report performance. For reviewing existing outputs, set `MODE='review'` and `EXISTING_RUN` to the saved run directory.

Training logic lives in `train.py`, audit/splitting in `data.py`, and inference in `predict.py`. The notebook calls those exact functions so the command-line and notebook paths stay synchronized.

In [ ]:
if MODE == 'review':
    if not EXISTING_RUN:
        raise ValueError('Set EXISTING_RUN to a completed Task 3 output directory.')
    RUN_DIR = Path(EXISTING_RUN).resolve()
else:
    RUN_DIR = run_experiment(ROOT, EXISTING_RUN, CONFIG)

print('Outputs:', RUN_DIR)
audit = json.loads((RUN_DIR / 'audit.json').read_text(encoding='utf-8'))
results = pd.read_csv(RUN_DIR / 'results_task3.csv')
selected = json.loads((RUN_DIR / 'selected_models.json').read_text(encoding='utf-8'))
provenance = json.loads((RUN_DIR / 'provenance.json').read_text(encoding='utf-8'))
if provenance['smoke_only']:
    display(Markdown('**SMOKE CHECK ONLY: these scores are not final experimental results.**'))

In [ ]:
display(pd.Series({key: value for key, value in audit.items() if not isinstance(value, (dict, list))}, name='measured_value').to_frame())
manifest = pd.read_csv(RUN_DIR / 'split_manifest.csv')
display(manifest['split'].value_counts().rename('rows').to_frame())
assert manifest.groupby('duplicate_group')['split'].nunique().max() == 1
coverage = pd.read_csv(RUN_DIR / 'class_coverage.csv')
display(coverage.pivot(index=['target', 'class'], columns='split', values='count').fillna(0).astype(int))

## Evaluation

Macro-F1 always uses the saved complete label vocabulary with `zero_division=0`, so a model cannot inflate its headline score by never predicting a difficult class. A second supported-class macro-F1 excludes labels with no true evaluation examples and is named explicitly. An absent class's zero is a bookkeeping convention, **not evidence that its real F1 was measured**.

There is no proven macro-F1 ceiling of 0.50. That number only follows from assuming four of eight class F1 scores are zero and the rest are perfect. Rare classes are difficult to estimate, not automatically unlearnable. Majority accuracy is a baseline, not a guaranteed lower bound for every model.

In [ ]:
display(results[['target', 'model', 'split', 'accuracy', 'macro_f1', 'macro_f1_supported', 'weighted_f1', 'n_evaluated', 'n_supported_classes']].round(4))

In [ ]:
for target in ('gender', 'usage'):
    display(Markdown(f'### {target}: selected {selected[target]["variant"]}'))
    display(pd.read_csv(RUN_DIR / f'{target}_final_holdout_report.csv', index_col=0).round(4))
    display(DisplayImage(filename=str(RUN_DIR / f'{target}_final_holdout_confusion.png')))
    display(DisplayImage(filename=str(RUN_DIR / f'{target}_{selected[target]["variant"]}_learning.png')))
    error_grid = RUN_DIR / f'{target}_final_holdout_errors.png'
    if error_grid.exists():
        display(DisplayImage(filename=str(error_grid)))

## Rare-Label Merging: A Separate Question

The exploratory mapping keeps Casual, Ethnic, Formal and Sports and maps Smart Casual, Travel, Party and Home to Other. It **does not** replace the eight-label final model or official CSV vocabulary.

For a fair comparison, sum the selected eight-class model's probabilities into the same five labels, then compare with a separately trained five-class model on the same row IDs and five-class truth. Comparing five-class against eight-class macro-F1 directly is invalid. Subtracting each one's majority baseline does **not** fix that different-task comparison either.

In [ ]:
merged = results.loc[results.target.eq('usage_5class')]
if len(merged):
    display(merged[['model', 'split', 'accuracy', 'macro_f1']].round(4))
else:
    print('Merged-label experiment was disabled for this run.')

## Discussion and Ultimate Judgement

The summary below is computed from actual outputs rather than declaring that every tuned model wins. Validation selects the MLP; holdout estimates its remaining generalization error. A high accuracy with poor minority recall should lead to a limited-use recommendation, not a deployment claim.

In [ ]:
for target in ('gender', 'usage'):
    final = results.loc[results.target.eq(target) & results.split.eq('holdout') & results.model.ne('majority')].iloc[0]
    baseline = results.loc[results.target.eq(target) & results.split.eq('holdout') & results.model.eq('majority')].iloc[0]
    report = pd.read_csv(RUN_DIR / f'{target}_final_holdout_report.csv', index_col=0)
    per_class = report.loc[selected[target]['classes']]
    observed = per_class.loc[per_class.support.gt(0)].sort_values('recall')
    display(Markdown(
        f'**{target}:** validation selected **{selected[target]["variant"]}**. '
        f'Holdout accuracy = **{final.accuracy:.3f}**, macro-F1 = **{final.macro_f1:.3f}** '
        f'(majority macro-F1 = {baseline.macro_f1:.3f}). '
        f'Macro-F1 change over this same-target baseline = **{final.macro_f1 - baseline.macro_f1:+.3f}**. '
        f'Lowest observed recall: **{observed.index[0]}**, {observed.iloc[0]["recall"]:.3f}, '
        f'support {int(observed.iloc[0]["support"])}.'))
    missing = per_class.loc[per_class.support.eq(0)].index.tolist()
    if missing:
        print('No holdout evidence for:', ', '.join(missing))

### What the Evidence Can and Cannot Say

- Read learning curves to distinguish underfitting from overfitting; input dimension alone proves neither. More MLP units can help but cannot add information absent from the photograph.
- Compare regularized versus weighted validation scores to assess the imbalance intervention. An improvement is not guaranteed. Small-class F1 can change substantially after only one different prediction.
- Exact duplicate grouping prevents one known leakage route, but near duplicates, common backgrounds and retailer biases remain. Target-conflicting groups routed to training mean holdout describes the cleaner, splittable groups, not every catalogue case.
- Retain all original gender/usage labels for the required task. The five-class experiment loses distinctions and cannot fill an eight-label submission honestly.
- A single split and seed do not demonstrate statistical superiority. Additional seeds/group-aware uncertainty estimates and independently labelled external photos are needed before a strong real-world recommendation.
- The current MLP is a reproducible educational prototype. The product's marketed gender and occasion are imperfectly specified by appearance; show alternative model scores and allow human correction.

## Independent Perspective: Published Work

[Liu et al., DeepFashion (CVPR 2016)](https://openaccess.thecvf.com/content_cvpr_2016/html/Liu_DeepFashion_Powering_Robust_CVPR_2016_paper.html) studies clothing recognition, attributes and retrieval with FashionNet and rich annotations. Its broader clothing-understanding objective is related, but its dataset, attributes, localization information and evaluation differ from these five gender/eight usage labels. We therefore compare design choices (spatial/landmark features versus our compact pixel MLP), **not headline accuracy values**. No pretrained weights from that work are used here.

[Zakizadeh et al. (2018)](https://arxiv.org/abs/1807.11674) investigates imbalanced/repetitive DeepFashion attribute annotations, scarce attributes and annotation cleanup. This supports investigating label quality and rare-label merging as research questions. It does not prove merging improves our task; our folded-versus-retrained experiment tests the local hypothesis without changing final labels.

**Remaining requirement:** these are related attribute studies, not matched gender/usage benchmarks. The internal holdout is not external data. To substantiate an independent real-world claim, obtain a separate, consented/licensed product-image set with retailer-provided gender/usage labels, freeze models first, exclude overlapping products and report the same per-class metrics. Do not invent external results or relabel photos from visual guesses.

## Save, Reload and Blind-Test Predictions

The final model and adjacent JSON save preprocessing dimensions and class order. `Task3Predictor` reloads them without fitting anything. The new prediction CSV preserves official IDs, column order, and non-Task-3 columns. It contains only this task's contribution; Tasks 1 and 2 must supply articleType and season later.

In [ ]:
MODELS_DIR = RUN_DIR / 'models'
predictor = Task3Predictor(MODELS_DIR)
example_path = DATASET / 'test/images_test' / f'{blind_template["id"].iloc[0]}.jpg'
display(pd.DataFrame(predictor.predict_image(example_path)).T[['label', 'score']])
submission_path = RUN_DIR / 'styles_prediction_task3.csv'
if not provenance['smoke_only']:
    if not submission_path.exists():
        submission = export_submission(ROOT, MODELS_DIR, submission_path)
    else:
        submission = pd.read_csv(submission_path)
    assert submission.id.equals(blind_template.id)
    assert list(submission.columns) == list(blind_template.columns)
    assert submission[['gender', 'usage']].notna().all().all()
    display(submission.head())
else:
    print('Blind-test export skipped for smoke models.')

## Image Upload Demo

This notebook widget calls the same saved-model inference API. Upload a product photograph to view its predicted catalogue labels. The displayed softmax scores are not calibrated confidence or proof of a person's identity. A combined team application remains separate from this Task 3 demonstration.

In [ ]:
upload_widget = notebook_demo(MODELS_DIR)

## Reproducibility and Handoff

The run contains a split manifest, configuration, source/data fingerprints, selected models, full-vocabulary metrics, per-class support, plots and predictions. Generated artifacts stay in the repo's already-ignored `outputs/` directory; include the selected models when preparing the assignment ZIP manually.

No `src` package is recreated, no teammate task is edited, and no commit or push is performed. Check the Task 3 README for commands and integration caveats. The group's report and external evaluation need team review before submission.

References: [TensorFlow EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping), [scikit-learn F1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html), and the two research papers linked above.

In [ ]:
display(pd.Series(provenance, name='run_provenance').to_frame())
print('Review these outputs before committing or sharing:', RUN_DIR)